##### How to format arguments into a single string column using format_string()?

##### format_string()
- Function **formats arguments** in a **printf-style** and returns the result as a **new string column** in a PySpark DataFrame.

**Syntax**

     format_string(format, *cols)

**format:**

**`printf-style Formatting:`**
- It uses standard **C printf** `style format specifiers` like
  - `%s for string`
  - `%d for integer`
- **`a) Formatting Numbers:`** You can `format numerical columns`, such as displaying a `floating-point number`.
  - `%f: Floating point number (e.g., %.2f for two decimal places)`
  -  `scientific notation with two decimal places using "%.2e"`

- **`b) Padding Integers:`** You can `left-pad an integer column` with `zeros` using format specifiers like **"%03d"** to ensure a minimum width.
  - `%03d: Integer left-padded with zeros to a width of 3 (e.g., 10 becomes 010)`

**`*cols:`**
- `One or more column names` or `pyspark.sql.Column objects` whose values will be inserted into the `format string`.
- The `number of columns` provided must match the `number of format specifiers` in the `format string`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, format_string

##### 1) format numerical columns

In [0]:
data = [(21, "Harish", 3567.0),
        (23, "Ravi", 5673.1),
        (22, "Rakesh", 4895.603456),
        (24, "Shiva", 9867.782345),
        (25, "Roshan", 7367.352345),
        (26, "Gupta", 6567.439345),
        (27, "Swaroop", 2467.912895)]

schema = ["Age", "Name", "Salary"]

df_frmt_01 = spark.createDataFrame(data, schema)
display(df_frmt_01)
df_frmt_01.printSchema()

Age,Name,Salary
21,Harish,3567.0
23,Ravi,5673.1
22,Rakesh,4895.603456
24,Shiva,9867.782345
25,Roshan,7367.352345
26,Gupta,6567.439345
27,Swaroop,2467.912895


root
 |-- Age: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Salary: double (nullable = true)



**If Salary is a float:**
- `2 decimal places`
- `Possibly leading zeros`
- `%03d is only for integers (d = decimal integer)`.

**Understanding %08.2f**

| Part | Meaning                    |
| ---- | -------------------------- |
| `%`  | Start format               |
| `0`  | Pad with zeros             |
| `8`  | Total width = 8 characters |
| `.2` | 2 decimal places           |
| `f`  | Floating-point             |

In [0]:
df_frmt_01.select("*", F.format_string('%03d', F.col("Age")).alias("formatted_pad_3d"),
                       F.format_string('%05d', F.col("Age")).alias("formatted_pad_5d")).display()

Age,Name,Salary,formatted_pad_3d,formatted_pad_5d
21,Harish,3567.0,021,00021
23,Ravi,5673.1,023,00023
22,Rakesh,4895.603456,022,00022
24,Shiva,9867.782345,024,00024
25,Roshan,7367.352345,025,00025
26,Gupta,6567.439345,026,00026
27,Swaroop,2467.912895,027,00027


In [0]:
df_frmt_01.select("Salary", F.format_string("%08.2f", col("Salary")).alias("formatted_salary_2f"),
                            F.format_string("%011.5f", col("Salary")).alias("formatted_salary_5f")).display()

Salary,formatted_salary_2f,formatted_salary_5f
3567.0,03567.00,03567.00000
5673.1,05673.10,05673.10000
4895.603456,04895.60,04895.60346
9867.782345,09867.78,09867.78235
7367.352345,07367.35,07367.35235
6567.439345,06567.44,06567.43935
2467.912895,02467.91,02467.91290


In [0]:
from pyspark.sql.functions import lpad

df_frmt_01.select("Salary",
    lpad(format_string("%.2f", col("Salary")), 10, "$").alias("formatted_salary_2f"),
    lpad(format_string("%.4f", col("Salary")), 11, "$").alias("formatted_salary_4f")).display()

Salary,formatted_salary_2f,formatted_salary_4f
3567.0,$$$3567.00,$$3567.0000
5673.1,$$$5673.10,$$5673.1000
4895.603456,$$$4895.60,$$4895.6035
9867.782345,$$$9867.78,$$9867.7823
7367.352345,$$$7367.35,$$7367.3523
6567.439345,$$$6567.44,$$6567.4393
2467.912895,$$$2467.91,$$2467.9129


In [0]:
df_frmt_01.select("*",
    F.format_string('%d %s %f', F.col("Age"), F.col("Name"), F.col("Salary")).alias("formatted_col"),
    F.format_string('%d; %s; %f', F.col("Age"), F.col("Name"), F.col("Salary")).alias("formatted_col_colon"),
    F.format_string('%d_%s_%.2f', F.col("Age"), F.col("Name"), F.col("Salary")).alias("formatted_col_float")).display()

Age,Name,Salary,formatted_col,formatted_col_colon,formatted_col_float
21,Harish,3567.0,21 Harish 3567.000000,21; Harish; 3567.000000,21_Harish_3567.00
23,Ravi,5673.1,23 Ravi 5673.100000,23; Ravi; 5673.100000,23_Ravi_5673.10
22,Rakesh,4895.603456,22 Rakesh 4895.603456,22; Rakesh; 4895.603456,22_Rakesh_4895.60
24,Shiva,9867.782345,24 Shiva 9867.782345,24; Shiva; 9867.782345,24_Shiva_9867.78
25,Roshan,7367.352345,25 Roshan 7367.352345,25; Roshan; 7367.352345,25_Roshan_7367.35
26,Gupta,6567.439345,26 Gupta 6567.439345,26; Gupta; 6567.439345,26_Gupta_6567.44
27,Swaroop,2467.912895,27 Swaroop 2467.912895,27; Swaroop; 2467.912895,27_Swaroop_2467.91


In [0]:
# Use format_string to create a new column based on existing column values
df_frmt_01.select(
    "*",
    F.format_string("Employee Name: %s & Age: %d", F.col("Name"), F.col("Age")).alias("formatted_col")
).display()

Age,Name,Salary,formatted_col
21,Harish,3567.0,Employee Name: Harish & Age: 21
23,Ravi,5673.1,Employee Name: Ravi & Age: 23
22,Rakesh,4895.603456,Employee Name: Rakesh & Age: 22
24,Shiva,9867.782345,Employee Name: Shiva & Age: 24
25,Roshan,7367.352345,Employee Name: Roshan & Age: 25
26,Gupta,6567.439345,Employee Name: Gupta & Age: 26
27,Swaroop,2467.912895,Employee Name: Swaroop & Age: 27


In [0]:
# Create a sample DataFrame
data = [("AF", 100111, "LM_BR", 101001, "CENTRAL"),
        ("LM", 100112, "PR_MN", 101002, "EXCHANGE"),
        ("BF", 100113, "SD_AQ", 101003, "DRYER"),
        ("PG", 100114, "WR_UY", 101004, "EURO"),
        ("YT", 100115, "TR_JK", 101005, "INR")]

columns = ["field1", "field2", "field3", "field4", "field5"]

df1 = spark.createDataFrame(data, columns)
display(df1)

field1,field2,field3,field4,field5
AF,100111,LM_BR,101001,CENTRAL
LM,100112,PR_MN,101002,EXCHANGE
BF,100113,SD_AQ,101003,DRYER
PG,100114,WR_UY,101004,EURO
YT,100115,TR_JK,101005,INR


In [0]:
# Use format_string to build a URL column
formatted_df = df1.withColumn(
    "input",
    F.format_string(
        "%s_%d_%s_%d_%s",
        F.col('field1'),
        F.col('field2'),
        F.col('field3'),
        F.col('field4'),
        F.col('field5')
    )
)

# Show the result
display(formatted_df)

field1,field2,field3,field4,field5,input
AF,100111,LM_BR,101001,CENTRAL,AF_100111_LM_BR_101001_CENTRAL
LM,100112,PR_MN,101002,EXCHANGE,LM_100112_PR_MN_101002_EXCHANGE
BF,100113,SD_AQ,101003,DRYER,BF_100113_SD_AQ_101003_DRYER
PG,100114,WR_UY,101004,EURO,PG_100114_WR_UY_101004_EURO
YT,100115,TR_JK,101005,INR,YT_100115_TR_JK_101005_INR


In [0]:
# Use format_string to build a URL column
formatted_df_11 = df1.withColumn(
    "input",
    F.format_string(
        "http://address.com",
        F.col('field1'),
        F.col('field2'),
        F.col('field3'),
        F.col('field4'),
        F.col('field5')
    )
).withColumn(
    "input_upd",
    F.format_string(
        "http://address.com_%s_%d_%s_%d_%s",
        F.col('field1'),
        F.col('field2'),
        F.col('field3'),
        F.col('field4'),
        F.col('field5')
    ))

# Show the result
display(formatted_df_11)

field1,field2,field3,field4,field5,input,input_upd
AF,100111,LM_BR,101001,CENTRAL,http://address.com,http://address.com_AF_100111_LM_BR_101001_CENTRAL
LM,100112,PR_MN,101002,EXCHANGE,http://address.com,http://address.com_LM_100112_PR_MN_101002_EXCHANGE
BF,100113,SD_AQ,101003,DRYER,http://address.com,http://address.com_BF_100113_SD_AQ_101003_DRYER
PG,100114,WR_UY,101004,EURO,http://address.com,http://address.com_PG_100114_WR_UY_101004_EURO
YT,100115,TR_JK,101005,INR,http://address.com,http://address.com_YT_100115_TR_JK_101005_INR


##### 2) Basic Scientific Notation

In [0]:
data = [(123456789.0,), (0.0001234,), (98765.4321,), (1000000000000.0,), (0.00000045,)]

df2 = spark.createDataFrame(data, ["value"])
display(df2)

value
1.23456789E8
1.234E-4
98765.4321
1.0E12
4.5E-7


In [0]:
df_deci = df2.select("value",
    format_string("%.2f", "value").alias("frmt_2_decimal"),
    format_string("%.2e", "value").alias("scientific_2_decimal"),
    format_string("%.5e", "value").alias("scientific_5_decimal")
)

display(df_deci)
df_deci.printSchema()

value,frmt_2_decimal,scientific_2_decimal,scientific_5_decimal
1.23456789E8,123456789.00,1.23e+08,1.23457e+08
1.234E-4,0.00,1.23e-04,1.23400e-04
98765.4321,98765.43,9.88e+04,9.87654e+04
1.0E12,1000000000000.00,1.00e+12,1.00000e+12
4.5E-7,0.00,4.50e-07,4.50000e-07


root
 |-- value: double (nullable = true)
 |-- frmt_2_decimal: string (nullable = false)
 |-- scientific_2_decimal: string (nullable = false)
 |-- scientific_5_decimal: string (nullable = false)



In [0]:
df2.select("value", *[format_string("%.2e", col("value").cast("float")).alias(c) for c in df2.columns]).display()

value,value
1.23456789E8,1.23e+08
1.234E-4,1.23e-04
98765.4321,9.88e+04
1.0E12,1.00e+12
4.5E-7,4.50e-07


##### 3) Using SQL Expression

In [0]:
from pyspark.sql.functions import expr

df_sql = df2.select(
    expr("format_string('%.3e', value)").alias("scientific")
)

display(df_sql)
df_sql.printSchema()

scientific
1.235e+08
1.234e-04
9.877e+04
1.000e+12
4.500e-07


root
 |-- scientific: string (nullable = false)

